Notebook to resolve the error below. It occurs when using GWHD of last SP form Sim1 as SHD for Sim2. Surprisingly this doesn't happen when Sim2 is run wiht iMOD5, but it happens when it's run with imod_python.

Simulation validation status:
    - imported_model model:
        - model options package:
        - ic package:
            - start:
                - nodata is not aligned with idomain
  File "G:\models\NBr\code\snakemake\NBr57.smk", line 95, in __rule_Mdl_Prep
  File "G:\code\WS_Mdl\imod\prep.py", line 53, in Sim
  File "G:\code\WS_Mdl\core\runtime.py", line 31, in timed_Exe
  File "G:\code\WS_Mdl\imod\prj.py", line 761, in PrSimP
  File "G:\code\WS_Mdl\core\runtime.py", line 31, in timed_Exe
  File "G:\.pixi\envs\default\Lib\site-packages\primod\metamod.py", line 98, in write
  File "G:\.pixi\envs\default\Lib\site-packages\imod\logging\logging_decorators.py", line 32, in wrapper
  File "G:\.pixi\envs\default\Lib\site-packages\imod\mf6\simulation.py", line 404, in write

In [1]:
from WS_Mdl.core import *
import imod
from WS_Mdl.xr.spatial import clip_Mdl_area
from matplotlib import pyplot as plt
from scipy.interpolate import griddata
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
MdlN_B = 'NBr52'
MdlN_S = 'NBr57'

In [3]:
M = Mdl_N(MdlN_B)

In [4]:
l_Pa = list((M.Pa.In / f"SHD/{MdlN_B}").glob("SHD_*.IDF"))

In [5]:
SHD = imod.idf.open(l_Pa, pattern='{name}_{time}_L{layer}_{MdlN}')

In [7]:
if SHD.y.values[0] > SHD.y.values[-1]:
    SHD = SHD.reindex(y=SHD.y[::-1])

In [ ]:
for L in SHD.layer[:]:
    if SHD.sel(layer=L).squeeze(drop=True).copy().isnull().all(): # If they're all null, repeat previous layer
        sprint(f"Layer {L} has no values. Repeating the previous layer.", style=warn)
        SHD_L = SHD_L.copy() # Just copy the previous layer again to maintain the same number of layers, even if they're all the same
        SHD_L = SHD_L.assign_coords(layer=L) # Otherwise it gets the previous L number...
    else: # Otherwise, use the current layer
        SHD_L = SHD.sel(layer=L).squeeze(drop=True).copy()
        #SHD_L.plot()
        # print('INITIAL')
        # plt.show()

        arr = SHD_L.to_numpy().copy()

        mask = np.isnan(arr)
        points = np.column_stack(np.where(~mask))
        values = arr[~mask]
        grid   = np.column_stack(np.where(mask))

        # Perform linear interpolation
        filled = griddata(points, values, grid, method='linear')

        # arr[mask] = filled
        # SHD_L.data = arr
        # SHD_L.plot()
        # print('LINEAR INTERPOLATION')
        # plt.show()

        # Perform nearest neighbor interpolation for remaining NaN values
        mask_rem = np.isnan(filled)
        filled[mask_rem] = griddata(points, values, grid[mask_rem], method='nearest')

        arr[mask] = filled
        SHD_L.data = arr

        # Collapse to y,x only so save doesn't append pattern per extra dims
        SHD_L.name = "SHD"
        SHD_L = SHD_L.assign_coords(time=pd.to_datetime(SHD.time.values[0]))

        # SHD_L.plot()
        # print('NEAREST NEIGHBOR INTERPOLATION FOR NAN VALUES')
        # plt.show()

    if np.isnan(SHD_L).any():
        sprint(f"Warning: NaN values remain in SHD_L for layer {L} after interpolation.", style=warn)

    out_dir = Path("G:/models/NBr/In/SHD/NBr57")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / f"SHD_{pd.to_datetime(SHD.time.values[0]).strftime('%Y%m%d')}_L{int(L)}_{MdlN_S}.idf"
    imod.idf.save(out_file, SHD_L, nodata=-999.9900, pattern=f"SHD_{{time:%Y%m%d}}_L{{layer}}_{MdlN_S}.idf")

    # for comparisson with from WS_Mdl.xr import fill_na; from WS_Mdl.xr import fill_na
    # SHD_L = SHD_L.broadcast_like(SHD.sel({'layer': L}))
    # SHD.sel(layer=L).data = SHD_L.data

Layer <xarray.DataArray 'layer' ()> Size: 8B
array(8)
Coordinates:
    layer    int64 8B 8
    dx       float64 8B 25.0
    dy       float64 8B -25.0 has no values. Repeating the previous layer.
Layer <xarray.DataArray 'layer' ()> Size: 8B
array(12)
Coordinates:
    layer    int64 8B 12
    dx       float64 8B 25.0
    dy       float64 8B -25.0 has no values. Repeating the previous layer.
Layer <xarray.DataArray 'layer' ()> Size: 8B
array(14)
Coordinates:
    layer    int64 8B 14
    dx       float64 8B 25.0
    dy       float64 8B -25.0 has no values. Repeating the previous layer.
Layer <xarray.DataArray 'layer' ()> Size: 8B
array(27)
Coordinates:
    layer    int64 8B 27
    dx       float64 8B 25.0
    dy       float64 8B -25.0 has no values. Repeating the previous layer.
Layer <xarray.DataArray 'layer' ()> Size: 8B
array(28)
Coordinates:
    layer    int64 8B 28
    dx       float64 8B 25.0
    dy       float64 8B -25.0 has no values. Repeating the previous layer.
Layer <xarray.Dat